In [1]:
import os
from dotenv import load_dotenv

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print('No API key was found!')
elif not api_key.startswith('sk-proj-'):
    print('An API key was found, but it doesnt start with "sk-proj-", i.e, it is not an OpenAI API key')
elif api_key.strip() != api_key:
    print('An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them.')
else:
    print('The API key found and looks good so far!')


The API key found and looks good so far!


# LangChain

In [3]:
import os
from typing import Optional

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable, RunnableParallel, RunnablePassthrough

# --- Configuration ---
# Ensure your API key environment variable is set (e.g., OPENAI_API_KEY)
try:
    llm: Optional[ChatOpenAI] = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
    if llm:
        print(f"Language model initialized: {llm.model_name}")
except Exception as e:
    print(f"Error initializing language model: {e}")
    llm = None


# --- Define Independent Chains ---
# These three chains represent distinct tasks that can be executed in parallel.

summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize the following topic concisely:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

questions_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Generate three interesting questions about the following topic:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

terms_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Identify 5-10 key terms from the following topic, separated by commas:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)


# --- Build the Parallel + Synthesis Chain ---

# 1. Define the block of tasks to run in parallel. The results of these,
#    along with the original topic, will be fed into the next step.
map_chain = RunnableParallel(
    {
        "summary": summarize_chain,
        "questions": questions_chain,
        "key_terms": terms_chain,
        "topic": RunnablePassthrough(),  # Pass the original topic through
    }
)

# 2. Define the final synthesis prompt which will combine the parallel results.
synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """Based on the following information:
     Summary: {summary}
     Related Questions: {questions}
     Key Terms: {key_terms}
     Synthesize a comprehensive answer."""),
    ("user", "Original topic: {topic}")
])

# 3. Construct the full chain by piping the parallel results directly
#    into the synthesis prompt, followed by the LLM and output parser.
full_parallel_chain = map_chain | synthesis_prompt | llm | StrOutputParser()


# --- Run the Chain ---
async def run_parallel_example(topic: str) -> None:
    """
    Asynchronously invokes the parallel processing chain with a specific topic
    and prints the synthesized result.

    Args:
        topic: The input topic to be processed by the LangChain chains.
    """
    if not llm:
        print("LLM not initialized. Cannot run example.")
        return

    print(f"\n--- Running Parallel LangChain Example for Topic: '{topic}' ---")
    try:
        # The input to `ainvoke` is the single 'topic' string, which is
        # then passed to each runnable in the `map_chain`.
        response = await full_parallel_chain.ainvoke(topic)
        print("\n--- Final Response ---")
        print(response)
    except Exception as e:
        print(f"\nAn error occurred during chain execution: {e}")

test_topic = "The history of space exploration"
# The notebook already runs an event loop, so await the coroutine directly.
await run_parallel_example(test_topic)

Language model initialized: gpt-4o-mini

--- Running Parallel LangChain Example for Topic: 'The history of space exploration' ---

--- Final Response ---
The history of space exploration is marked by significant milestones, technological advancements, and international collaboration, shaping our understanding of the universe and our place within it. 

**Beginnings of Space Exploration:**
Space exploration officially began in the mid-20th century with the launch of the Soviet satellite Sputnik on October 4, 1957. This event not only marked the dawn of the Space Age but also ignited the Space Race between the United States and the Soviet Union during the Cold War. The competition spurred rapid advancements in technology and human spaceflight capabilities. Notable achievements included Yuri Gagarin's historic orbit of Earth in 1961, making him the first human in space, and the U.S. Apollo program, which successfully landed astronauts on the Moon in 1969 with Apollo 11. This period was cha

## LangChain + Ollama

In [4]:
# Same parallel chain, running on a local model served by Ollama instead of OpenAI.
# Prerequisites: install Ollama from https://ollama.com, then pull the model:
#   ollama pull llama3.1:8b

from typing import Optional

from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import Runnable, RunnableParallel, RunnablePassthrough

OLLAMA_MODEL = "llama3.1:8b"  # any model from `ollama list`, e.g. qwen2.5:7b or mistral:7b

# Ollama runs locally, so no API key is needed.
# Note ChatOllama exposes `.model`, not the `.model_name` ChatOpenAI uses.
llm: Optional[ChatOllama] = ChatOllama(model=OLLAMA_MODEL, temperature=0.7)
print(f"Language model initialized: {llm.model}")


# --- Define Independent Chains ---
# These three chains represent distinct tasks that can be executed in parallel.

summarize_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Summarize the following topic concisely:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

questions_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Generate three interesting questions about the following topic:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)

terms_chain: Runnable = (
    ChatPromptTemplate.from_messages([
        ("system", "Identify 5-10 key terms from the following topic, separated by commas:"),
        ("user", "{topic}")
    ])
    | llm
    | StrOutputParser()
)


# --- Build the Parallel + Synthesis Chain ---
# Identical to the cell above: the pattern does not change with the provider.
map_chain = RunnableParallel(
    {
        "summary": summarize_chain,
        "questions": questions_chain,
        "key_terms": terms_chain,
        "topic": RunnablePassthrough(),
    }
)

synthesis_prompt = ChatPromptTemplate.from_messages([
    ("system", """Based on the following information:
     Summary: {summary}
     Related Questions: {questions}
     Key Terms: {key_terms}
     Synthesize a comprehensive answer."""),
    ("user", "Original topic: {topic}")
])

full_parallel_chain = map_chain | synthesis_prompt | llm | StrOutputParser()


# --- Run the Chain ---
async def run_parallel_example(topic: str) -> None:
    """Asynchronously invokes the parallel chain and prints the synthesized result."""
    print(f"\n--- Running Parallel LangChain Example for Topic: '{topic}' ---")
    response = await full_parallel_chain.ainvoke(topic)
    print("\n--- Final Response ---")
    print(response)

test_topic = "The history of space exploration"
# The notebook already runs an event loop, so await the coroutine directly.
await run_parallel_example(test_topic)

Language model initialized: llama3.1:8b

--- Running Parallel LangChain Example for Topic: 'The history of space exploration' ---

--- Final Response ---
The history of space exploration is a rich and fascinating topic that spans several decades, with numerous milestones and achievements that have shaped our understanding of the universe. From the early years of space exploration in the 1950s and 1960s to the present day, the journey to space has been marked by numerous breakthroughs, setbacks, and innovations.

**Early Years (1950s-1960s)**

The early years of space exploration began with the launch of Sputnik 1, the first artificial satellite, by the Soviet Union in 1957. This event marked the beginning of the space age and sparked a sense of competition between the Soviet Union and the United States to explore space. The Soviet Union's Yuri Gagarin became the first human in space in 1961, followed by the first American in space, Alan Shepard, and the first American in orbit, John Gl

# Google ADK

In [5]:
# Part of agent.py --> Follow https://google.github.io/adk-docs/get-started/quickstart/ to learn the setup
# Requires GOOGLE_API_KEY in .env (loaded above).

from google.adk.agents import LlmAgent, ParallelAgent, SequentialAgent
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.genai import types

GEMINI_MODEL = "gemini-flash-latest"

# --- 1. Define Researcher Sub-Agents (to run in parallel) ---

# Researcher 1: Renewable Energy
researcher_agent_1 = LlmAgent(
    name="RenewableEnergyResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in energy.
Research the latest advancements in 'renewable energy sources'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches renewable energy sources.",
    tools=[google_search],
    # Store result in state for the merger agent
    output_key="renewable_energy_result"
)

# Researcher 2: Electric Vehicles
researcher_agent_2 = LlmAgent(
    name="EVResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in transportation.
Research the latest developments in 'electric vehicle technology'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches electric vehicle technology.",
    tools=[google_search],
    # Store result in state for the merger agent
    output_key="ev_technology_result"
)

# Researcher 3: Carbon Capture
researcher_agent_3 = LlmAgent(
    name="CarbonCaptureResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in climate solutions.
Research the current state of 'carbon capture methods'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches carbon capture methods.",
    tools=[google_search],
    # Store result in state for the merger agent
    output_key="carbon_capture_result"
)

# --- 2. Create the ParallelAgent (Runs researchers concurrently) ---
# This agent orchestrates the concurrent execution of the researchers.
# It finishes once all researchers have completed and stored their results in state.
parallel_research_agent = ParallelAgent(
    name="ParallelWebResearchAgent",
    sub_agents=[researcher_agent_1, researcher_agent_2, researcher_agent_3],
    description="Runs multiple research agents in parallel to gather information."
)

# --- 3. Define the Merger Agent (Runs *after* the parallel agents) ---
# This agent takes the results stored in the session state by the parallel agents
# and synthesizes them into a single, structured response with attributions.
merger_agent = LlmAgent(
    name="SynthesisAgent",
    model=GEMINI_MODEL,  # Or potentially a more powerful model if needed for synthesis
    instruction="""You are an AI Assistant responsible for combining research findings into a structured report.

Your primary task is to synthesize the following research summaries, clearly attributing findings to their source areas. Structure your response using headings for each topic. Ensure the report is coherent and integrates the key points smoothly.

**Crucially: Your entire response MUST be grounded *exclusively* on the information provided in the 'Input Summaries' below. Do NOT add any external knowledge, facts, or details not present in these specific summaries.**

**Input Summaries:**

*   **Renewable Energy:**
    {renewable_energy_result}

*   **Electric Vehicles:**
    {ev_technology_result}

*   **Carbon Capture:**
    {carbon_capture_result}

**Output Format:**

## Summary of Recent Sustainable Technology Advancements

### Renewable Energy Findings
(Based on RenewableEnergyResearcher's findings)
[Synthesize and elaborate *only* on the renewable energy input summary provided above.]

### Electric Vehicle Findings
(Based on EVResearcher's findings)
[Synthesize and elaborate *only* on the EV input summary provided above.]

### Carbon Capture Findings
(Based on CarbonCaptureResearcher's findings)
[Synthesize and elaborate *only* on the carbon capture input summary provided above.]

### Overall Conclusion
[Provide a brief (1-2 sentence) concluding statement that connects *only* the findings presented above.]

Output *only* the structured report following this format. Do not include introductory or concluding phrases outside this structure, and strictly adhere to using only the provided input summary content.
""",
    description="Combines research findings from parallel agents into a structured, cited report, strictly grounded on provided inputs.",
    # No tools needed for merging
    # No output_key needed here, as its direct response is the final output of the sequence
)


# --- 4. Create the SequentialAgent (Orchestrates the overall flow) ---
# This is the main agent that will be run. It first executes the ParallelAgent
# to populate the state, and then executes the MergerAgent to produce the final output.
sequential_pipeline_agent = SequentialAgent(
    name="ResearchAndSynthesisPipeline",
    # Run parallel research first, then merge
    sub_agents=[parallel_research_agent, merger_agent],
    description="Coordinates parallel research and synthesizes the results."
)

root_agent = sequential_pipeline_agent

# --- 5. Run the pipeline ---
APP_NAME = "parallel_research_app"
USER_ID = "user_1"

runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

message = types.Content(
    role="user",
    parts=[types.Part(text="Research the latest sustainable technology advancements.")],
)

async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message
):
    if event.is_final_response() and event.content and event.content.parts:
        print(f"--- {event.author} ---")
        print(event.content.parts[0].text)
        print()


/var/folders/_n/81b2c8ld4ybgj_p8m3_g6rc00000gn/T/ipykernel_82487/1525665019.py:64: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  parallel_research_agent = ParallelAgent(
/var/folders/_n/81b2c8ld4ybgj_p8m3_g6rc00000gn/T/ipykernel_82487/1525665019.py:123: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  sequential_pipeline_agent = SequentialAgent(
Direct use of automatic function calling (AFC) in AsyncModels.generate_content is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message. Similarly, direct use of AFC in AsyncModels.generate_content_stream is not recommended. Instead, we recommend to use AFC in AsyncChat.send_message_stream.


--- CarbonCaptureResearcher ---
Current carbon capture methods primarily rely on point-source capture using liquid solvents and emerging Direct Air Capture (DAC) systems that remove ambient $\text{CO}_2$ via chemical sorbents and mineralization. While technological innovations are rapidly improving capture efficiency, high energy requirements, high costs, and infrastructure constraints remain the main barriers to large-scale deployment.

--- EVResearcher ---
Recent developments in electric vehicle technology are centered on the commercialization of solid-state and sodium-ion batteries, which offer significantly higher energy densities, faster charging times, and greater safety compared to conventional lithium-ion cells. Additionally, the expansion of ultra-fast charging infrastructure and bidirectional Vehicle-to-Grid (V2G) systems is transforming EVs into efficient, flexible assets for sustainable energy management.

--- RenewableEnergyResearcher ---
Recent breakthroughs in renewable 

## Google ADK + Ollama

In [6]:
# Same parallel research pipeline, running on a local model served by Ollama through LiteLLM.
# Prerequisites:
#   ollama pull qwen2.5:7b
#
# One substitution is unavoidable: ADK's built-in `google_search` is Gemini-only
# server-side grounding. Attaching it to any other model raises
#   ValueError: Google search tool is not supported for model ollama_chat/qwen2.5:7b
# so the researchers below call a local mock search tool instead. The pattern being
# taught - three agents working concurrently, then a merger - is unchanged; only the
# source of the facts is.

import os

from google.adk.agents import LlmAgent, ParallelAgent, SequentialAgent
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

# LiteLLM needs to know where the local Ollama server listens.
os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")

# Use the 'ollama_chat' provider, not 'ollama': it is the one that handles tool
# calling correctly. Tool use needs a capable model; qwen2.5:7b manages it.
OLLAMA_MODEL = "ollama_chat/qwen2.5:7b"


def ollama_model() -> LiteLlm:
    """Each agent needs its own model instance."""
    return LiteLlm(model=OLLAMA_MODEL)


def local_search(topic: str) -> str:
    """Simulates a web search for the given topic and returns a short finding.

    Args:
        topic: The subject to look up.

    Returns:
        A one-line research finding for the topic.
    """
    print(f"TOOL: local_search('{topic}')")
    # Matched on keywords rather than the exact string: a 7B model often rewrites
    # the topic it was told to pass, so exact-match lookups come back empty.
    canned = [
        (("renewable", "solar", "wind", "photovoltaic"),
         "Perovskite-silicon tandem cells have passed 33% lab efficiency, and "
         "grid-scale battery storage costs keep falling."),
        (("electric vehicle", "ev ", " ev", "battery", "sodium"),
         "Sodium-ion packs have entered volume production as a cheaper, "
         "cobalt-free alternative for short-range vehicles."),
        (("carbon", "capture", "direct air", "emission"),
         "Direct air capture plants are scaling to megatonne designs, but cost "
         "per tonne remains the main obstacle."),
    ]
    haystack = topic.lower()
    for keywords, finding in canned:
        if any(k in haystack for k in keywords):
            return finding
    return f"No local results for '{topic}'."


# --- 1. Define Researcher Sub-Agents (to run in parallel) ---

researcher_agent_1 = LlmAgent(
    name="RenewableEnergyResearcher",
    model=ollama_model(),
    instruction="""You are an AI Research Assistant specializing in energy.
Call the local_search tool with exactly this topic, unchanged: "renewable energy sources".
Summarize what it returns in one sentence.
Output *only* the summary.
""",
    description="Researches renewable energy sources.",
    tools=[local_search],
    output_key="renewable_energy_result",
)

researcher_agent_2 = LlmAgent(
    name="EVResearcher",
    model=ollama_model(),
    instruction="""You are an AI Research Assistant specializing in transportation.
Call the local_search tool with exactly this topic, unchanged: "electric vehicle technology".
Summarize what it returns in one sentence.
Output *only* the summary.
""",
    description="Researches electric vehicle technology.",
    tools=[local_search],
    output_key="ev_technology_result",
)

researcher_agent_3 = LlmAgent(
    name="CarbonCaptureResearcher",
    model=ollama_model(),
    instruction="""You are an AI Research Assistant specializing in climate solutions.
Call the local_search tool with exactly this topic, unchanged: "carbon capture methods".
Summarize what it returns in one sentence.
Output *only* the summary.
""",
    description="Researches carbon capture methods.",
    tools=[local_search],
    output_key="carbon_capture_result",
)

# --- 2. Create the ParallelAgent (runs the researchers concurrently) ---
parallel_research_agent = ParallelAgent(
    name="ParallelWebResearchAgent",
    sub_agents=[researcher_agent_1, researcher_agent_2, researcher_agent_3],
    description="Runs multiple research agents in parallel to gather information.",
)

# --- 3. Define the Merger Agent (runs *after* the parallel agents) ---
# The instruction is shortened from the cell above: a 7B model follows a compact
# format far more reliably than the long, heavily emphasised original.
merger_agent = LlmAgent(
    name="SynthesisAgent",
    model=ollama_model(),
    instruction="""Combine the three research findings below into a short report.
Use only the information given. Do not add outside knowledge.

Renewable Energy: {renewable_energy_result}
Electric Vehicles: {ev_technology_result}
Carbon Capture: {carbon_capture_result}

Write one short paragraph per topic under these headings:
### Renewable Energy Findings
### Electric Vehicle Findings
### Carbon Capture Findings
### Overall Conclusion
""",
    description="Combines research findings into a structured, grounded report.",
)

# --- 4. Create the SequentialAgent (orchestrates the overall flow) ---
sequential_pipeline_agent = SequentialAgent(
    name="ResearchAndSynthesisPipeline",
    sub_agents=[parallel_research_agent, merger_agent],
    description="Coordinates parallel research and synthesizes the results.",
)

root_agent = sequential_pipeline_agent

# --- 5. Run the pipeline ---
APP_NAME = "parallel_research_ollama_app"
USER_ID = "user_1"

runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

message = types.Content(
    role="user",
    parts=[types.Part(text="Research the latest sustainable technology advancements.")],
)

async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message
):
    if event.is_final_response() and event.content and event.content.parts:
        text = event.content.parts[0].text
        if text:
            print(f"--- {event.author} ---")
            print(text)
            print()

/var/folders/_n/81b2c8ld4ybgj_p8m3_g6rc00000gn/T/ipykernel_82487/93445082.py:104: DeprecationWarning: ParallelAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  parallel_research_agent = ParallelAgent(
/var/folders/_n/81b2c8ld4ybgj_p8m3_g6rc00000gn/T/ipykernel_82487/93445082.py:133: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  sequential_pipeline_agent = SequentialAgent(
/Users/hodung/Workspace/ta/agentic-design-patterns/.venv/lib/python3.12/site-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()


TOOL: local_search('carbon capture methods')
TOOL: local_search('electric vehicle technology')
TOOL: local_search('renewable energy sources')
--- CarbonCaptureResearcher ---
Direct air capture plants are scaling to megatonne designs, but cost per tonne remains the main obstacle.

--- EVResearcher ---
Sodium-ion packs have entered volume production as a cheaper, cobalt-free alternative for short-range vehicles.

--- RenewableEnergyResearcher ---
Perovskite-silicon tandem cells have passed 33% lab efficiency, and grid-scale battery storage costs keep falling.

--- SynthesisAgent ---
### Renewable Energy Findings
Perovskite-silicon tandem cells have achieved a lab efficiency of over 33%, and the costs for grid-scale battery storage continue to decrease, making these technologies more viable for widespread adoption.

### Electric Vehicle Findings
Sodium-ion battery packs are now in mass production, offering a cost-effective and cobalt-free alternative for short-range electric vehicles.

##

In [7]:
# Same pipeline, written with the graph-based Workflow API (google-adk 2.x).
# ParallelAgent / SequentialAgent are deprecated in favor of Workflow.
# Requires GOOGLE_API_KEY in .env (loaded above).

from google.adk import Agent, Workflow
from google.adk.runners import InMemoryRunner
from google.adk.tools import google_search
from google.adk.workflow import JoinNode
from google.genai import types

GEMINI_MODEL = "gemini-flash-latest"

# --- 1. Define Researcher Agents (nodes that run in parallel) ---

researcher_agent_1 = Agent(
    name="RenewableEnergyResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in energy.
Research the latest advancements in 'renewable energy sources'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches renewable energy sources.",
    tools=[google_search],
)

researcher_agent_2 = Agent(
    name="EVResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in transportation.
Research the latest developments in 'electric vehicle technology'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches electric vehicle technology.",
    tools=[google_search],
)

researcher_agent_3 = Agent(
    name="CarbonCaptureResearcher",
    model=GEMINI_MODEL,
    instruction="""You are an AI Research Assistant specializing in climate solutions.
Research the current state of 'carbon capture methods'.
Use the Google Search tool provided.
Summarize your key findings concisely (1-2 sentences).
Output *only* the summary.
""",
    description="Researches carbon capture methods.",
    tools=[google_search],
)

# --- 2. Join Node (waits for all three researchers, then fans their outputs in) ---
# Its output is a dict keyed by the name of each upstream node.
gather_research = JoinNode(name="GatherResearch")

# --- 3. Function Node (formats the joined outputs for the merger) ---
TOPICS = {
    "RenewableEnergyResearcher": "Renewable Energy",
    "EVResearcher": "Electric Vehicles",
    "CarbonCaptureResearcher": "Carbon Capture",
}


def format_summaries(node_input: dict[str, str]) -> str:
    """Turn the joined researcher outputs into a labeled 'Input Summaries' block."""
    return "\n\n".join(
        f"*   **{topic}:**\n    {node_input[name]}" for name, topic in TOPICS.items()
    )


# --- 4. Define the Merger Agent (the final node of the graph) ---
merger_agent = Agent(
    name="SynthesisAgent",
    model=GEMINI_MODEL,  # Or potentially a more powerful model if needed for synthesis
    instruction="""You are an AI Assistant responsible for combining research findings into a structured report.

Your primary task is to synthesize the research summaries you receive as input, clearly attributing findings to their source areas. Structure your response using headings for each topic. Ensure the report is coherent and integrates the key points smoothly.

**Crucially: Your entire response MUST be grounded *exclusively* on the information provided in the input summaries. Do NOT add any external knowledge, facts, or details not present in these specific summaries.**

**Output Format:**

## Summary of Recent Sustainable Technology Advancements

### Renewable Energy Findings
(Based on RenewableEnergyResearcher's findings)
[Synthesize and elaborate *only* on the renewable energy input summary provided above.]

### Electric Vehicle Findings
(Based on EVResearcher's findings)
[Synthesize and elaborate *only* on the EV input summary provided above.]

### Carbon Capture Findings
(Based on CarbonCaptureResearcher's findings)
[Synthesize and elaborate *only* on the carbon capture input summary provided above.]

### Overall Conclusion
[Provide a brief (1-2 sentence) concluding statement that connects *only* the findings presented above.]

Output *only* the structured report following this format. Do not include introductory or concluding phrases outside this structure, and strictly adhere to using only the provided input summary content.
""",
    description="Combines research findings from the parallel nodes into a structured, cited report, strictly grounded on provided inputs.",
)

# --- 5. Build the Workflow graph ---
# START fans out to the three researchers, which fan back in through the join
# node; the joined result is formatted and handed to the merger agent.
root_agent = Workflow(
    name="ResearchAndSynthesisPipeline",
    edges=[
        (
            "START",
            (researcher_agent_1, researcher_agent_2, researcher_agent_3),
            gather_research,
        ),
        (gather_research, format_summaries, merger_agent),
    ],
)

# --- 6. Run the workflow ---
APP_NAME = "workflow_research_app"
USER_ID = "user_1"

runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

message = types.Content(
    role="user",
    parts=[types.Part(text="Research the latest sustainable technology advancements.")],
)

async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message
):
    if event.is_final_response() and event.content and event.content.parts:
        text = event.content.parts[0].text
        if text:
            print(f"--- {event.author} ---")
            print(text)
            print()


--- RenewableEnergyResearcher ---
Recent advancements in renewable energy focus on high-efficiency perovskite tandem solar cells, next-generation solid-state and flow battery storage, and expanding offshore wind systems. Furthermore, the integration of AI-driven smart grids and scalable green hydrogen production is significantly improving grid stability and facilitating the decarbonization of heavy industries.

--- CarbonCaptureResearcher ---
Current carbon capture methods are transitioning from traditional point-source chemical absorption to next-generation technologies like Direct Air Capture (DAC) and advanced materials, including metal-organic frameworks (MOFs) and solid sorbents. While commercial deployment and geological storage projects are scaling globally, widespread adoption remains constrained by significant energy penalties, infrastructure demands, and high operational costs.

--- EVResearcher ---
Recent advancements in electric vehicle technology focus on the development o

## Google ADK Workflow + Ollama

In [8]:
# Same Workflow pipeline, running on a local model served by Ollama through LiteLLM.
# Prerequisites:
#   ollama pull qwen2.5:7b
#
# As in the ADK cell above, `google_search` cannot be attached to a non-Gemini model,
# so the researchers call the same local mock search tool. Everything else is the
# graph from the cell above: a tuple of nodes fans out, a JoinNode fans back in.

import os

from google.adk import Agent, Workflow
from google.adk.workflow import JoinNode
from google.adk.models.lite_llm import LiteLlm
from google.adk.runners import InMemoryRunner
from google.genai import types

# LiteLLM needs to know where the local Ollama server listens.
os.environ.setdefault("OLLAMA_API_BASE", "http://localhost:11434")

# Use the 'ollama_chat' provider, not 'ollama': it is the one that handles tool
# calling correctly. Tool use needs a capable model; qwen2.5:7b manages it.
OLLAMA_MODEL = "ollama_chat/qwen2.5:7b"


def ollama_model() -> LiteLlm:
    """Each agent needs its own model instance."""
    return LiteLlm(model=OLLAMA_MODEL)


def local_search(topic: str) -> str:
    """Simulates a web search for the given topic and returns a short finding.

    Args:
        topic: The subject to look up.

    Returns:
        A one-line research finding for the topic.
    """
    print(f"TOOL: local_search('{topic}')")
    # Matched on keywords rather than the exact string: a 7B model often rewrites
    # the topic it was told to pass, so exact-match lookups come back empty.
    canned = [
        (("renewable", "solar", "wind", "photovoltaic"),
         "Perovskite-silicon tandem cells have passed 33% lab efficiency, and "
         "grid-scale battery storage costs keep falling."),
        (("electric vehicle", "ev ", " ev", "battery", "sodium"),
         "Sodium-ion packs have entered volume production as a cheaper, "
         "cobalt-free alternative for short-range vehicles."),
        (("carbon", "capture", "direct air", "emission"),
         "Direct air capture plants are scaling to megatonne designs, but cost "
         "per tonne remains the main obstacle."),
    ]
    haystack = topic.lower()
    for keywords, finding in canned:
        if any(k in haystack for k in keywords):
            return finding
    return f"No local results for '{topic}'."


# --- 1. Define Researcher Agents (nodes that run in parallel) ---

researcher_agent_1 = Agent(
    name="RenewableEnergyResearcher",
    model=ollama_model(),
    instruction="""You are an AI Research Assistant specializing in energy.
Call the local_search tool with exactly this topic, unchanged: "renewable energy sources".
Summarize what it returns in one sentence.
Output *only* the summary.
""",
    description="Researches renewable energy sources.",
    tools=[local_search],
)

researcher_agent_2 = Agent(
    name="EVResearcher",
    model=ollama_model(),
    instruction="""You are an AI Research Assistant specializing in transportation.
Call the local_search tool with exactly this topic, unchanged: "electric vehicle technology".
Summarize what it returns in one sentence.
Output *only* the summary.
""",
    description="Researches electric vehicle technology.",
    tools=[local_search],
)

researcher_agent_3 = Agent(
    name="CarbonCaptureResearcher",
    model=ollama_model(),
    instruction="""You are an AI Research Assistant specializing in climate solutions.
Call the local_search tool with exactly this topic, unchanged: "carbon capture methods".
Summarize what it returns in one sentence.
Output *only* the summary.
""",
    description="Researches carbon capture methods.",
    tools=[local_search],
)

# --- 2. Join Node (waits for all three researchers, then fans their outputs in) ---
# Its output is a dict keyed by the name of each upstream node.
gather_research = JoinNode(name="GatherResearch")

# --- 3. Function Node (formats the joined outputs for the merger) ---
TOPICS = {
    "RenewableEnergyResearcher": "Renewable Energy",
    "EVResearcher": "Electric Vehicles",
    "CarbonCaptureResearcher": "Carbon Capture",
}


def format_summaries(node_input: dict[str, str]) -> str:
    """Turn the joined researcher outputs into a labeled 'Input Summaries' block."""
    return "\n\n".join(
        f"*   **{topic}:**\n    {node_input[name]}" for name, topic in TOPICS.items()
    )


# --- 4. Define the Merger Agent (the final node of the graph) ---
# Instruction kept compact: a 7B model follows a short format far more reliably.
merger_agent = Agent(
    name="SynthesisAgent",
    model=ollama_model(),
    instruction="""Combine the research findings you receive into a short report.
Use only the information given. Do not add outside knowledge.

Write one short paragraph per topic under these headings:
### Renewable Energy Findings
### Electric Vehicle Findings
### Carbon Capture Findings
### Overall Conclusion
""",
    description="Combines research findings into a structured, grounded report.",
)

# --- 5. Build the Workflow graph ---
root_agent = Workflow(
    name="ResearchAndSynthesisPipeline",
    edges=[
        (
            "START",
            (researcher_agent_1, researcher_agent_2, researcher_agent_3),
            gather_research,
        ),
        (gather_research, format_summaries, merger_agent),
    ],
)

# --- 6. Run the workflow ---
APP_NAME = "workflow_research_ollama_app"
USER_ID = "user_1"

runner = InMemoryRunner(agent=root_agent, app_name=APP_NAME)
session = await runner.session_service.create_session(app_name=APP_NAME, user_id=USER_ID)

message = types.Content(
    role="user",
    parts=[types.Part(text="Research the latest sustainable technology advancements.")],
)

async for event in runner.run_async(
    user_id=USER_ID, session_id=session.id, new_message=message
):
    if event.is_final_response() and event.content and event.content.parts:
        text = event.content.parts[0].text
        if text:
            print(f"--- {event.author} ---")
            print(text)
            print()

TOOL: local_search('carbon capture methods')
TOOL: local_search('renewable energy sources')
TOOL: local_search('electric vehicle technology')
--- CarbonCaptureResearcher ---
Direct air capture plants are scaling to megatonne designs, but the main obstacle remains the cost per tonne.

--- RenewableEnergyResearcher ---
Perovskite-silicon tandem cells have passed 33% lab efficiency, and grid-scale battery storage costs keep falling.

--- EVResearcher ---
Sodium-ion packs have entered volume production as a cheaper, cobalt-free alternative for short-range electric vehicles.

--- SynthesisAgent ---
### Renewable Energy Findings
Perovskite-silicon tandem cells have achieved a lab efficiency of 33%, marking a significant advancement in photovoltaic technology. Additionally, the costs of grid-scale battery storage continue to decrease, enhancing the feasibility of renewable energy solutions.

### Electric Vehicle Findings
Sodium-ion packs have entered mass production, offering a cost-effective